In [1]:
import pandas as pd

In [2]:
# Load data
data = pd.read_csv("../data/raw/world_cities.csv")
data.head()

,country,name,lat,lng
0,AD,les Escaldes,42.50729,1.53414
1,AD,Andorra la Vella,42.50779,1.52109
2,AE,Warīsān,25.16744,55.40708
3,AE,Umm Suqaym,25.15491,55.21015
4,AE,Umm Al Quwain City,25.56473,55.55517


In [3]:
data.shape

(34135, 4)

In [4]:
# Filter countries for Europe and North America only
europe = ["AL","AD","AT","BY","BE","BA","BG","HR","CY","CZ","DK","EE","FI","FR",
          "DE","GR","HU","IS","IE","IT","LV","LI","LT","LU","MT","MD","MC","ME",
          "NL","MK","NO","PL","PT","RO","RS","SK","SI","ES","SE","CH","UA","GB"]

north_america = ["US","CA","MX"]

target_countries = europe + north_america
target_cities = data[data["country"].isin(target_countries)]

In [5]:
# Random sample with sample size = 300
cities = target_cities.sample(n=300, random_state=42)
cities.shape

(300, 4)

In [6]:
# Randomly assign growth potential score to each city
import numpy as np
np.random.seed(42)

cities["growth_potential"]=np.random.normal(loc=50, scale=15, size=len(cities))
cities[["name","growth_potential"]].head()

,name,growth_potential
23735,Grimstad,57.450712
31176,Lexington,47.926035
8248,Weißensee,59.715328
14064,Dún Laoghaire,72.845448
31098,Huntington,46.487699


In [7]:
# Create test period in months (datetimeindex)
months = pd.date_range(start="2025-01-01", end="2026-03-01", freq="MS")
# Months Dataframe
months_df = pd.DataFrame({"month":months})
months_df.head()

,month
0,2025-01-01
1,2025-02-01
2,2025-03-01
3,2025-04-01
4,2025-05-01


In [8]:
# Assign rollout months by growth potential (6 groups)

cities["rollout_wave"] = pd.qcut(cities["growth_potential"], q=6, labels=[5,4,3,2,1,0])


In [9]:
cities[["name","growth_potential","rollout_wave"]].sort_values("growth_potential",ascending=False).head(10)

,name,growth_potential,rollout_wave
11668,Châlons-en-Champagne,107.790972,0
31015,Niles,90.802537,0
29375,Vidradnyi,86.948632,0
1564,Sevlievo,84.719879,0
32188,Gardena,82.856834,0
25039,Lubartów,82.159161,0
4458,Saint John West,81.995501,0
10193,Maó,81.832343,0
29064,Volochysk,78.451895,0
30997,Maywood,78.292789,0


In [10]:
# Map rollout wave to actual month using dictionary 

rollout_months = {
    0: pd.Timestamp("2025-07-01"),
    1: pd.Timestamp("2025-08-01"),
    2: pd.Timestamp("2025-09-01"),
    3: pd.Timestamp("2025-10-01"),
    4: pd.Timestamp("2025-11-01"),
    5: pd.Timestamp("2025-12-01"),
}

cities["go_live_date"] = cities["rollout_wave"].map(rollout_months)
cities[["name", "growth_potential", "rollout_wave", "go_live_date"]].head(10)

,name,growth_potential,rollout_wave,go_live_date
23735,Grimstad,57.450712,1,2025-08-01
31176,Lexington,47.926035,3,2025-10-01
8248,Weißensee,59.715328,1,2025-08-01
14064,Dún Laoghaire,72.845448,0,2025-07-01
31098,Huntington,46.487699,3,2025-10-01
32174,Fair Oaks,46.487946,3,2025-10-01
31423,Dover,73.688192,0,2025-07-01
32278,Monterey,61.511521,1,2025-08-01
30246,Webster Groves,42.957884,4,2025-11-01
4245,Brossard,58.138401,1,2025-08-01


In [11]:
# Cross join to create panel
panel = cities.merge(months_df, how="cross")
panel.shape
panel.head()

,country,name,lat,lng,growth_potential,rollout_wave,go_live_date,month
0,NO,Grimstad,58.3405,8.59343,57.450712,1,2025-08-01,2025-01-01
1,NO,Grimstad,58.3405,8.59343,57.450712,1,2025-08-01,2025-02-01
2,NO,Grimstad,58.3405,8.59343,57.450712,1,2025-08-01,2025-03-01
3,NO,Grimstad,58.3405,8.59343,57.450712,1,2025-08-01,2025-04-01
4,NO,Grimstad,58.3405,8.59343,57.450712,1,2025-08-01,2025-05-01


In [12]:
# Add flag for treated groups - FALSE before go live date

panel["treated"] = panel["month"] >= panel["go_live_date"]
panel[["name","month","go_live_date","treated"]].sample(10)

,name,month,go_live_date,treated
414,Moose Jaw,2025-10-01,2025-08-01,True
3388,Karcag,2026-02-01,2025-07-01,True
144,Brossard,2025-10-01,2025-08-01,True
1183,Ottawa,2026-02-01,2025-09-01,True
3075,Rákospalota,2025-01-01,2025-11-01,False
387,Cornwall,2026-01-01,2025-09-01,True
4191,Steyr,2025-07-01,2025-10-01,False
2586,Pradl,2025-07-01,2025-10-01,False
2743,Zacapu,2026-02-01,2025-12-01,True
1024,Tavares,2025-05-01,2025-09-01,False


In [13]:
# Set retention rates based on growth potential and base retention rate
base_retention = 0.25
growth_effect = (panel["growth_potential"]-50)*0.002
panel["baseline_outcome"]=base_retention+growth_effect
panel[["name", "growth_potential", "baseline_outcome"]].sort_values("growth_potential", ascending=False).head(5)

,name,growth_potential,baseline_outcome
3142,Châlons-en-Champagne,107.790972,0.365582
3143,Châlons-en-Champagne,107.790972,0.365582
3136,Châlons-en-Champagne,107.790972,0.365582
3137,Châlons-en-Champagne,107.790972,0.365582
3138,Châlons-en-Champagne,107.790972,0.365582


In [14]:
# Introduce Seasonal Impact

seasonal_effect_by_month = {
    1: 0.02,   # January resurgence
    2: 0.00,
    3: 0.00,
    4: 0.00,
    5: -0.01,
    6: -0.03,  # summer lull begins
    7: -0.04,  # summer lull
    8: -0.03,  # summer lull / back-to-school prep
    9: 0.03,   # back-to-school bump
    10: 0.01,
    11: 0.00,
    12: -0.01, # holidays
}

panel["month_num"]=panel["month"].dt.month
panel["seasonal_effect"]=panel["month_num"].map(seasonal_effect_by_month)
panel["outcome_no_treatment"]= panel["baseline_outcome"]+ panel["seasonal_effect"]
panel[["name", "month", "baseline_outcome", "seasonal_effect", "outcome_no_treatment"]].sample(8)

,name,month,baseline_outcome,seasonal_effect,outcome_no_treatment
1087,Rinkeby,2025-08-01,0.248925,-0.03,0.218925
695,Waltham,2025-06-01,0.236181,-0.03,0.206181
1431,Békéscsaba,2025-07-01,0.206095,-0.04,0.166095
2612,Gateshead,2025-03-01,0.258301,0.00,0.258301
4460,Maspalomas,2025-06-01,0.259219,-0.03,0.229219
3283,Minden,2026-02-01,0.235439,0.00,0.235439
3021,Guadalupe,2025-07-01,0.266824,-0.04,0.226824
1536,Santutxu,2025-07-01,0.239719,-0.04,0.199719


In [15]:
# Add the treatment effect 
true_effect = 0.04 # +0.04 percentage points

panel["outcome"] = panel["outcome_no_treatment"] + (panel["treated"]*true_effect)
panel[["name", "month", "treated", "outcome_no_treatment", "outcome"]].sample(8)

,name,month,treated,outcome_no_treatment,outcome
4222,Blackheath,2025-08-01,True,0.239864,0.279864
1305,Coburg,2025-01-01,False,0.279863,0.279863
1711,Sønderborg,2025-02-01,False,0.244229,0.244229
2900,Guimarães,2025-06-01,False,0.182628,0.182628
327,Maroúsi,2026-01-01,True,0.263227,0.303227
2394,Lysychansk,2025-10-01,True,0.279697,0.319697
2735,Zacapu,2025-06-01,False,0.187873,0.187873
4337,Melito di Napoli,2025-03-01,False,0.231319,0.231319


In [16]:
# Spillover: Part 1 - Distance Between Cities
# Test

def haversine_km(lat1,lng1,lat2,lng2):
    R =6371 #Earth's radius in km
    lat1, lng1, lat2, lng2 = map(np.radians, [lat1,lng1,lat2,lng2])
    dlat = lat2 - lat1
    dlng = lng2 - lng1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlng/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# test on first 2 cities
city_a = cities.iloc[0]
city_b = cities.iloc[1]

distance = haversine_km(city_a["lat"], city_a["lng"], city_b["lat"], city_b["lng"])
print(city_a["country"], city_a["name"], "to", city_b["country"],city_b["name"], ":", distance, "km")
cities.head()
# test successful

NO Grimstad to US Lexington : 5558.487482998077 km


,country,name,lat,lng,growth_potential,rollout_wave,go_live_date
23735,NO,Grimstad,58.34050,8.59343,57.450712,1,2025-08-01
31176,US,Lexington,42.44732,-71.22450,47.926035,3,2025-10-01
8248,DE,Weißensee,52.55632,13.46649,59.715328,1,2025-08-01
14064,IE,Dún Laoghaire,53.29395,-6.13586,72.845448,0,2025-07-01
31098,US,Huntington,40.88310,-85.49748,46.487699,3,2025-10-01


In [17]:
# Apply distance calculation for all cities vis cross join
city_pairs = cities.merge(cities, how="cross", suffixes=("_a","_b"))
city_pairs = city_pairs[city_pairs["name_a"] != city_pairs["name_b"]]

city_pairs["distance_km"] = haversine_km(
    city_pairs["lat_a"], city_pairs["lng_a"],
    city_pairs["lat_b"], city_pairs["lng_b"]
)

city_pairs[["name_a", "name_b", "distance_km"]].sort_values("distance_km").head(10)

,name_a,name_b,distance_km
23834,Zürich (Kreis 6) / Unterstrass,Zürich (Kreis 6),0.602786
40279,Zürich (Kreis 6),Zürich (Kreis 6) / Unterstrass,0.602786
49174,Porta,la Prosperitat,0.905618
82363,la Prosperitat,Porta,0.905618
63659,Ciutat Vella,Eixample,1.359794
17912,Eixample,Ciutat Vella,1.359794
48923,Porta,La Sagrera,1.477749
7063,La Sagrera,Porta,1.477749
8642,La Courneuve,Aubervilliers,1.579591
72628,Aubervilliers,La Courneuve,1.579591


In [18]:
# Apply Decay calculation - further the city from a target city the lower the spillover impact

decay_distance = 100  # km

city_pairs["decay_weight"] = np.exp(-city_pairs["distance_km"] / decay_distance)
city_pairs[["name_a", "name_b", "distance_km", "decay_weight"]].sort_values("distance_km").head(10)

,name_a,name_b,distance_km,decay_weight
23834,Zürich (Kreis 6) / Unterstrass,Zürich (Kreis 6),0.602786,0.993990
40279,Zürich (Kreis 6),Zürich (Kreis 6) / Unterstrass,0.602786,0.993990
49174,Porta,la Prosperitat,0.905618,0.990985
82363,la Prosperitat,Porta,0.905618,0.990985
63659,Ciutat Vella,Eixample,1.359794,0.986494
17912,Eixample,Ciutat Vella,1.359794,0.986494
48923,Porta,La Sagrera,1.477749,0.985331
7063,La Sagrera,Porta,1.477749,0.985331
8642,La Courneuve,Aubervilliers,1.579591,0.984328
72628,Aubervilliers,La Courneuve,1.579591,0.984328


In [19]:
# Spillover calculation
spillover_calc = city_pairs.merge(
    panel[["name", "month", "treated"]],
    left_on = "name_a", right_on="name",
    how="left"
)

spillover_calc = spillover_calc.rename(columns={"treated": "treated_a"})
spillover_calc = spillover_calc.drop(columns=["name"])

spillover_calc.shape
spillover_calc.head()

,country_a,name_a,lat_a,lng_a,growth_potential_a,rollout_wave_a,go_live_date_a,country_b,name_b,lat_b,lng_b,growth_potential_b,rollout_wave_b,go_live_date_b,distance_km,decay_weight,month,treated_a
0,NO,Grimstad,58.3405,8.59343,57.450712,1,2025-08-01,US,Lexington,42.44732,-71.2245,47.926035,3,2025-10-01,5558.487483,7.240951e-25,2025-01-01,False
1,NO,Grimstad,58.3405,8.59343,57.450712,1,2025-08-01,US,Lexington,42.44732,-71.2245,47.926035,3,2025-10-01,5558.487483,7.240951e-25,2025-02-01,False
2,NO,Grimstad,58.3405,8.59343,57.450712,1,2025-08-01,US,Lexington,42.44732,-71.2245,47.926035,3,2025-10-01,5558.487483,7.240951e-25,2025-03-01,False
3,NO,Grimstad,58.3405,8.59343,57.450712,1,2025-08-01,US,Lexington,42.44732,-71.2245,47.926035,3,2025-10-01,5558.487483,7.240951e-25,2025-04-01,False
4,NO,Grimstad,58.3405,8.59343,57.450712,1,2025-08-01,US,Lexington,42.44732,-71.2245,47.926035,3,2025-10-01,5558.487483,7.240951e-25,2025-05-01,False


In [20]:
# Aggregate spillover to one calculation - add the decay weight and treated_a into single signal

#Weighted signal
spillover_calc["weighted_signal"] = spillover_calc["decay_weight"] * spillover_calc["treated_a"]

spillover_scores = (
    spillover_calc.groupby(["name_b", "month"])["weighted_signal"]
    .sum()
    .reset_index()
)

spillover_scores = spillover_scores.rename(columns={"weighted_signal": "raw_spillover_score"})

spillover_scores.shape
spillover_scores.head(10)


,name_b,month,raw_spillover_score
0,Achim,2025-01-01,0.000000
1,Achim,2025-02-01,0.000000
2,Achim,2025-03-01,0.000000
3,Achim,2025-04-01,0.000000
4,Achim,2025-05-01,0.000000
5,Achim,2025-06-01,0.000000
6,Achim,2025-07-01,0.826269
7,Achim,2025-08-01,0.956513
8,Achim,2025-09-01,1.949507
9,Achim,2025-10-01,2.415962


In [21]:
spillover_scores.describe()

,month,raw_spillover_score
count,4500,4500.000000
mean,2025-08-01 04:48:00,1.208482
min,2025-01-01 00:00:00,0.000000
25%,2025-04-01 00:00:00,0.000000
50%,2025-08-01 00:00:00,0.220446
75%,2025-12-01 00:00:00,1.842018
max,2026-03-01 00:00:00,8.299340
std,NaN,1.796949


In [22]:
# Calibrating saturating transform
spillover_scores = spillover_scores.merge(panel[["name","month","treated"]], left_on=["name_b","month"], right_on=["name","month"], how="left")
spillover_scores = spillover_scores.drop(columns=["name"])

cap = 0.015    # 1.5 percentage points, as a decimal fraction — same scale as true_effect (0.04)
scale = 2

spillover_scores["spillover_bump"] = cap * (1-np.exp(-spillover_scores["raw_spillover_score"]/ scale))
spillover_scores["spillover_bump"] = spillover_scores["spillover_bump"]*(1-spillover_scores["treated"])

spillover_scores[["name_b", "month", "raw_spillover_score", "treated", "spillover_bump"]].sample(10)

,name_b,month,raw_spillover_score,treated,spillover_bump
3812,Sylvania,2025-03-01,0.000000,False,0.000000
4180,Waltham,2025-11-01,4.543922,True,0.000000
3006,Popasna,2025-07-01,0.002464,False,0.000018
4069,Vibo Valentia,2025-05-01,0.000000,False,0.000000
589,Châlons-en-Champagne,2025-05-01,0.000000,False,0.000000
857,Dorchester,2025-03-01,0.000000,False,0.000000
4166,Volos,2025-12-01,0.306930,True,0.000000
680,Cloverdale,2025-06-01,0.000000,False,0.000000
2107,Marina di Carrara,2025-08-01,0.837069,False,0.005130
3755,Streetsboro,2025-06-01,0.000000,False,0.000000


In [23]:
panel = panel.merge(spillover_scores[["name_b","month","spillover_bump"]], left_on=["name","month"], right_on=["name_b","month"], how="left")
panel["outcome"] = panel["outcome"]+panel["spillover_bump"]

panel.head(10)

,country,name,lat,lng,growth_potential,rollout_wave,go_live_date,month,treated,baseline_outcome,month_num,seasonal_effect,outcome_no_treatment,outcome,name_b,spillover_bump
0,NO,Grimstad,58.3405,8.59343,57.450712,1,2025-08-01,2025-01-01,False,0.264901,1,0.02,0.284901,0.284901,Grimstad,0.000000
1,NO,Grimstad,58.3405,8.59343,57.450712,1,2025-08-01,2025-02-01,False,0.264901,2,0.00,0.264901,0.264901,Grimstad,0.000000
2,NO,Grimstad,58.3405,8.59343,57.450712,1,2025-08-01,2025-03-01,False,0.264901,3,0.00,0.264901,0.264901,Grimstad,0.000000
3,NO,Grimstad,58.3405,8.59343,57.450712,1,2025-08-01,2025-04-01,False,0.264901,4,0.00,0.264901,0.264901,Grimstad,0.000000
4,NO,Grimstad,58.3405,8.59343,57.450712,1,2025-08-01,2025-05-01,False,0.264901,5,-0.01,0.254901,0.254901,Grimstad,0.000000
5,NO,Grimstad,58.3405,8.59343,57.450712,1,2025-08-01,2025-06-01,False,0.264901,6,-0.03,0.234901,0.234901,Grimstad,0.000000
6,NO,Grimstad,58.3405,8.59343,57.450712,1,2025-08-01,2025-07-01,False,0.264901,7,-0.04,0.224901,0.224940,Grimstad,0.000039
7,NO,Grimstad,58.3405,8.59343,57.450712,1,2025-08-01,2025-08-01,True,0.264901,8,-0.03,0.234901,0.274901,Grimstad,0.000000
8,NO,Grimstad,58.3405,8.59343,57.450712,1,2025-08-01,2025-09-01,True,0.264901,9,0.03,0.294901,0.334901,Grimstad,0.000000
9,NO,Grimstad,58.3405,8.59343,57.450712,1,2025-08-01,2025-10-01,True,0.264901,10,0.01,0.274901,0.314901,Grimstad,0.000000


In [34]:
# Create user level data

user_numbers = pd.DataFrame({"user_num": range(1, 501)})
rct_users = user_numbers.merge(cities, how="cross")

# Assign variatn randomly
np.random.seed(42)
rct_users["variant"] = np.random.choice(["A","B"], size=len(rct_users))
rct_users["variant"].value_counts()

variant
B    75035
A    74965
Name: count, dtype: int64

In [36]:
# Assign each user a retention probability
baseline_retention = 0.25
growth_effect = (rct_users["growth_potential"] - 50) * 0.002
rct_users["baseline_prob"] = baseline_retention + growth_effect

nested_effect = 0.02
rct_users["true_prob"] = rct_users["baseline_prob"] + (rct_users["variant"] == "B") * nested_effect

In [37]:
rct_users[["name", "variant", "baseline_prob", "true_prob"]].sample(10)

,name,variant,baseline_prob,true_prob
29628,Dunkirk,B,0.228541,0.248541
55712,Ciutat Vella,A,0.278620,0.278620
44141,Streetsboro,B,0.255141,0.275141
68645,Ashland,B,0.239965,0.259965
135221,Palomeras Bajas,A,0.193982,0.193982
105571,Nogent-sur-Oise,B,0.206924,0.226924
135904,Huntington,B,0.242975,0.262975
138928,La Courneuve,A,0.231981,0.231981
39763,Porta,B,0.225380,0.245380
12609,Brossard,B,0.266277,0.286277


In [43]:
# Simulate binary outcome - Turn each users true_prob into retained flag
np.random.seed(42)
rct_users["retained"] = np.random.binomial(1, rct_users["true_prob"])

# Checks
rct_users.groupby("variant")["retained"].mean()
rct_users.shape
rct_users.isna().sum()

# Save as CSV

rct_users.to_csv("../data/raw/rct_users.csv", index=False)